# Exploration Smoke Test

This notebook exercises the current public API with local data only.


In [1]:
from portafolios import (
    Universe,
    local_loader,
    EqualWeightConstructor,
    Markowitz,
    NaiveRiskParity,
    HRPStyle,
    HRPRecursive,
    Backtester,
    MonteCarloEngine,
    PortfolioVisualizer,
)

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "portafolios").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH = PROJECT_ROOT / "data" / "processed" / "data_clean_stock_data.csv"
YF_SNAPSHOT_PATH = PROJECT_ROOT / "data" / "yf_snapshot.csv"

print("Project root:", PROJECT_ROOT)
print("Processed CSV exists:", CSV_PATH.exists())
print("Yahoo snapshot exists:", YF_SNAPSHOT_PATH.exists())


Project root: C:\Users\narro\Desktop\semestre\honores_actuaria
Processed CSV exists: True
Yahoo snapshot exists: True


In [2]:
u = Universe(
    universe_name="exploration_smoke",
    loader=local_loader,
    loader_kwargs={"path": YF_SNAPSHOT_PATH, "prefer_adj_close": True},
    tickers=["AAPL", "MSFT", "AMZN"],
    start="2024-01-02",
    end="2024-03-28",
    construction_start="2024-01-02",
    construction_end="2024-02-15",
    auto_save_data=False,
).prepare_data()

u.prices.head()


,AAPL,MSFT,AMZN
Date,,,
2024-01-02,183.731308,364.589417,149.929993
2024-01-03,182.355606,364.324036,148.470001
2024-01-04,180.039658,361.709106,144.570007
2024-01-05,179.317154,361.522278,145.240005
2024-01-08,183.652145,368.344757,149.100006


In [3]:
u.build(EqualWeightConstructor())
u.build(Markowitz(), ret_kind="simple", allow_short=False)
u.build(NaiveRiskParity())
u.build(HRPStyle())
u.build(HRPRecursive())
u.list_constructions()


['equal_weight',
 'markowitz_max_sharpe',
 'naive_risk_parity',
 'hrp_style',
 'hrp_recursive']

In [4]:
results = Backtester.run_all(u)
results["hrp_style"].summary_metrics


{'n_periods': 30,
 'total_return': 0.00927262980041621,
 'annualized_return': 0.08061593018409186,
 'annualized_volatility': 0.16354761331334305,
 'sharpe_ratio': 0.49292024842721927,
 'max_drawdown': -0.0391562652091102}

In [5]:
bt = Backtester(u, "hrp_style")
bt.summarize_window(start_date="2024-02-16", end_date="2024-03-15")


{'n_periods': 21,
 'total_return': -0.003526970282428432,
 'annualized_return': -0.04151221118321191,
 'annualized_volatility': 0.17847421120507947,
 'sharpe_ratio': -0.23259501136279823,
 'max_drawdown': -0.0391562652091102,
 'window_start': '2024-02-16 00:00:00',
 'window_end': '2024-03-15 00:00:00'}

In [6]:
MonteCarloEngine.run_all(u, horizon=20, n_simulations=25, seed=7)


{'equal_weight': MonteCarloResult(construction_name='equal_weight', horizon=20, n_simulations=25, simulated_paths=         sim_0     sim_1     sim_2     sim_3     sim_4     sim_5     sim_6  \
 step                                                                         
 0     1.000000  1.000000  1.000000  1.000000  1.000000  1.000000  1.000000   
 1     1.002025  1.010705  1.003436  1.008938  0.998458  0.991387  1.020593   
 2     0.995343  1.017943  1.008863  1.010379  1.006589  0.994390  1.009923   
 3     1.003909  1.018648  1.023245  1.017040  0.989735  0.999023  1.012018   
 4     0.999204  1.029413  1.033966  0.992748  0.992645  1.006861  1.011431   
 5     0.985049  1.035413  1.029904  1.005683  0.997939  1.004094  1.024314   
 6     0.962705  1.050094  1.045060  0.996313  1.005512  1.009665  1.043560   
 7     0.966764  1.063811  1.033714  1.018188  1.026224  1.026185  1.059026   
 8     0.969711  1.071955  1.041802  1.035517  1.034912  1.026910  1.065098   
 9     0.989565  1

In [7]:
viz = PortfolioVisualizer(u)
viz.plot_backtest("hrp_style", start_date="2024-02-16", end_date="2024-03-28")
